<a href="https://colab.research.google.com/github/np03cs4a240035-commits/Library-Management/blob/main/library_managment_system.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
import os
import json
from datetime import datetime, timedelta

# ── Data Storage ──────────────────────────────────────────────────────────────
DATA_FILE = "library_data.json"

def load_data():
    if os.path.exists(DATA_FILE):
        with open(DATA_FILE, "r") as f:
            return json.load(f)
    return {"books": {}, "members": {}, "issued": {}}

def save_data(data):
    with open(DATA_FILE, "w") as f:
        json.dump(data, f, indent=2)

# ── Book Functions ─────────────────────────────────────────────────────────────
def add_book(data):
    print("\n Add Book ")
    book_id = input("Book ID: ").strip()
    if book_id in data["books"]:
        print("Book ID already exists!")
        return
    title  = input("Title: ").strip()
    author = input("Author: ").strip()
    genre  = input("Genre: ").strip()
    copies = int(input("Number of copies: "))
    data["books"][book_id] = {
        "title": title, "author": author,
        "genre": genre, "copies": copies, "available": copies
    }
    save_data(data)
    print(f"Book '{title}' added successfully!")

def view_books(data):
    print("\n All Books ")
    if not data["books"]:
        print("No books found.")
        return
    print(f"{'ID':<8} {'Title':<25} {'Author':<20} {'Genre':<15} {'Available':<10}")
    print("-" * 80)
    for bid, b in data["books"].items():
        print(f"{bid:<8} {b['title']:<25} {b['author']:<20} {b['genre']:<15} {b['available']}/{b['copies']}")

def search_book(data):
    print("\n Search Book ")
    keyword = input("Enter title or author to search: ").strip().lower()
    found = [(bid, b) for bid, b in data["books"].items()
             if keyword in b["title"].lower() or keyword in b["author"].lower()]
    if not found:
        print("No matching books found.")
        return
    print(f"\n{'ID':<8} {'Title':<25} {'Author':<20} {'Available'}")
    print("-" * 65)
    for bid, b in found:
        print(f"{bid:<8} {b['title']:<25} {b['author']:<20} {b['available']}/{b['copies']}")

def delete_book(data):
    print("\n Delete Book ")
    book_id = input("Book ID to delete: ").strip()
    if book_id not in data["books"]:
        print("Book not found!")
        return
    # Check if any copy is currently issued
    issued_ids = [v["book_id"] for v in data["issued"].values()]
    if book_id in issued_ids:
        print("Cannot delete — some copies are currently issued!")
        return
    title = data["books"][book_id]["title"]
    del data["books"][book_id]
    save_data(data)
    print(f"Book '{title}' deleted.")

#  Member Functions (done by Nirbhaya Thapa)

# Add new members with validation
# members added based on their id
def add_member(data):
    print("\n Add Member ")
    member_id = input("Member ID: ").strip()
    if member_id in data["members"]:
        print("Member ID already exists!")
        return
    name  = input("Name: ").strip()
    email = input("Email: ").strip()
    phone = input("Phone: ").strip()
    data["members"][member_id] = {"name": name, "email": email, "phone": phone}
    save_data(data)
    print(f"Member '{name}' added successfully!")

# Done by Nirbhaya Thapa
# Display all members
# members viewed based on their ID
def view_members(data):
    print("\n All Members ")
    if not data["members"]:
        print("No members found.")
        return
    print(f"{'ID':<10} {'Name':<20} {'Email':<25} {'Phone'}")
    print("-" * 70)
    for mid, m in data["members"].items():
        print(f"{mid:<10} {m['name']:<20} {m['email']:<25} {m['phone']}")

# Issue / Return Functions (Done by Nirbhaya Thapa)
# Books issued based on member ID
# Issue logic with availability check
def issue_book(data):
    print("\n Issue Book ")
    member_id = input("Member ID: ").strip()
    if member_id not in data["members"]:
        print("Member not found!")
        return
    book_id = input("Book ID: ").strip()
    if book_id not in data["books"]:
        print("Book not found!")
        return
    if data["books"][book_id]["available"] <= 0:
        print("No copies available right now!")
        return
    issue_id  = f"ISS{len(data['issued'])+1:04d}"
    issue_date = datetime.now().strftime("%Y-%m-%d")
    due_date   = (datetime.now() + timedelta(days=14)).strftime("%Y-%m-%d")
    data["issued"][issue_id] = {
        "member_id": member_id, "book_id": book_id,
        "issue_date": issue_date, "due_date": due_date, "returned": False
    }
    data["books"][book_id]["available"] -= 1
    save_data(data)
    print(f"Book issued! Issue ID: {issue_id} | Due: {due_date}")

# Done by Nirbhaya Thapa
# Books returned by entering issue ID and member ID
# Return logic with fine calculation
def return_book(data):
    print("\n Return Book ")
    issue_id = input("Issue ID: ").strip()
    if issue_id not in data["issued"]:
        print("Issue record not found!")
        return
    record = data["issued"][issue_id]
    if record["returned"]:
        print("This book was already returned.")
        return
    return_date = datetime.now()
    due_date    = datetime.strptime(record["due_date"], "%Y-%m-%d")
    fine = max(0, (return_date - due_date).days) * 5  # Rs. 5/day fine
    record["returned"] = True
    record["return_date"] = return_date.strftime("%Y-%m-%d")
    data["books"][record["book_id"]]["available"] += 1
    save_data(data)
    print(f"Book returned successfully!")
    if fine > 0:
        print(f"Late fine: Rs. {fine} ({(return_date - due_date).days} days overdue)")

def view_issued(data):
    print("\n Issued Books ")
    active = [(iid, r) for iid, r in data["issued"].items() if not r["returned"]]
    if not active:
        print("No books currently issued.")
        return
    print(f"{'Issue ID':<10} {'Member':<15} {'Book ID':<10} {'Due Date':<12} {'Status'}")
    print("-" * 65)
    today = datetime.now().date()
    for iid, r in active:
        due   = datetime.strptime(r["due_date"], "%Y-%m-%d").date()
        status = "OVERDUE" if today > due else "On Time"
        mname  = data["members"].get(r["member_id"], {}).get("name", r["member_id"])
        print(f"{iid:<10} {mname:<15} {r['book_id']:<10} {r['due_date']:<12} {status}")

# ── Main Menu ──────────────────────────────────────────────────────────────────
def main():
    data = load_data()
    menu = """
╔══════════════════════════════════╗
║    📚 LIBRARY MANAGEMENT SYSTEM  ║
╠══════════════════════════════════╣
║  1. Add Book                     ║
║  2. View All Books               ║
║  3. Search Book                  ║
║  4. Delete Book                  ║
║  5. Add Member                   ║
║  6. View All Members             ║
║  7. Issue Book                   ║
║  8. Return Book                  ║
║  9. View Issued Books            ║
║  0. Exit                         ║
╚══════════════════════════════════╝"""
    actions = {
        "1": add_book,    "2": view_books,   "3": search_book,
        "4": delete_book, "5": add_member,   "6": view_members,
        "7": issue_book,  "8": return_book,  "9": view_issued,
    }
    while True:
        print(menu)
        choice = input("Enter choice (then press Enter): ").strip()
        if choice == "0":
            print(" Goodbye!")
            break
        elif choice in actions:
            actions[choice](data)
        else:
            print("Invalid choice. Try again.")

if __name__ == "__main__":
    main()


╔══════════════════════════════════╗
║    📚 LIBRARY MANAGEMENT SYSTEM  ║
╠══════════════════════════════════╣
║  1. Add Book                     ║
║  2. View All Books               ║
║  3. Search Book                  ║
║  4. Delete Book                  ║
║  5. Add Member                   ║
║  6. View All Members             ║
║  7. Issue Book                   ║
║  8. Return Book                  ║
║  9. View Issued Books            ║
║  0. Exit                         ║
╚══════════════════════════════════╝


KeyboardInterrupt: Interrupted by user